In [ ]:
import numpy as np
import math

SIZE = 5
q_table = np.full((SIZE, SIZE, 4), .5)
GOAL = (SIZE-1,SIZE-1)
EPISODES = 3000
STEP_SIZE = .15
DISCOUNT = .9
SHOW_EPISODE = 500

def main():
    epsilon = 1
    average = 0

    for current in range(EPISODES):
        #print(current)
        pos = (0,0)
        count = 0

        while pos != GOAL:
            a = pick_action(pos,epsilon)
            action = (round(math.cos(a)),round(math.sin(a)))
            new_pos = find_new(action,pos)

            current_q = max(q_table[pos[0]][pos[1]])
            reward = find_reward(new_pos)
            max_future_q = max(q_table[new_pos[0]][new_pos[1]])
            new_q = (1-STEP_SIZE)*current_q + STEP_SIZE*(reward+DISCOUNT*max_future_q)
            q_table[pos[0]][pos[1]][int(a/(np.pi/2))] = new_q

            pos = new_pos
            count += 1

        if((current+1)%SHOW_EPISODE==0):
            #print_episode(q_table)
            print(average)
        else:
            average = (average*(current%SHOW_EPISODE)+count)/((current+1)%SHOW_EPISODE)
        epsilon = epsilon - epsilon/(EPISODES//2-1)

    print_q_table()
    print_episode(q_table)


def pick_action(pos,epsilon):
    if(np.random.random()<epsilon):
        return(np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    else:
        return((np.pi/2)*np.argmax(q_table[pos[0]][pos[1]]))

def find_new(action,pos):
    temp1 = action[0]+pos[0]
    if(temp1>=SIZE):
        temp1 = SIZE-1
    elif(temp1<0):
        temp1 = 0
    temp2 = action[1]+pos[1]
    if(temp2>=SIZE):
        temp2 = SIZE-1
    elif(temp2<0):
        temp2 = 0
    return(temp1,temp2)

def find_reward(new_pos):
    if(GOAL==new_pos):
        return 1
    else:
        return 0

def print_episode(q):
    for y in range(SIZE):
        line = ""
        for x in range(SIZE):
            a = np.argmax(q[x][y])
            if(a==0):
                line += "> "
            elif(a==1):
                line += "V "
            elif(a==2):
                line += "< "
            else:
                line += "^ "
        print(line+"\n")

def print_q_table():
    for y in range(SIZE):
        print("[", end="")
        for x in range(SIZE-1):
            print(np.round(q_table[x][y],2), end="")
            print(" ", end="")
        print(np.round(q_table[SIZE-1][y],2), end="")
        print("]")


main()

47.226452905811655
19.350701402805612
14.224448897795591
11.515030060120235
10.162324649298602
9.49098196392786
[[0.69 0.69 0.68 0.68] [0.77 0.77 0.75 0.76] [0.86 0.86 0.83 0.84] [0.95 0.95 0.92 0.94] [1.04 1.06 1.03 1.04]]
[[0.77 0.77 0.76 0.75] [0.86 0.86 0.83 0.83] [0.95 0.95 0.92 0.92] [1.06 1.06 1.03 1.03] [1.16 1.17 1.14 1.14]]
[[0.86 0.86 0.84 0.83] [0.95 0.95 0.92 0.92] [1.06 1.06 1.03 1.03] [1.17 1.17 1.14 1.14] [1.29 1.3  1.27 1.27]]
[[0.95 0.95 0.94 0.92] [1.06 1.06 1.03 1.03] [1.17 1.17 1.14 1.14] [1.3  1.3  1.27 1.27] [1.43 1.45 1.41 1.41]]
[[1.06 1.04 1.04 1.03] [1.17 1.16 1.14 1.14] [1.3  1.29 1.27 1.27] [1.45 1.43 1.41 1.41] [0.5 0.5 0.5 0.5]]
> > > > V 

> > > > V 

> > > > V 

> > > > V 

> > > > > 



In [25]:
import numpy as np
import math
import sys

SIZE = 5
DEGREE = 2
GOAL = np.array([(4,0),(0,4),(2,2),(3,3),(4,4),(1,1)])
EPISODES = 10000
STEP_SIZE = .2
STEP_SIZE_DECAY = .01
DISCOUNT = .9
SHOW_EPISODE = 100


def main():
    epsilon = 1
    average = 0
    step_size = STEP_SIZE
    # [x0y0, x0y1, x0y2, x1y0, x1y1, x1y2, x2y0, x2y1, x2y2]
    weights = np.full(((DEGREE+1),(DEGREE+1)), 0.0)

    for current in range(EPISODES):
        pos = (0,0)
        count = 0

        while(find_reward(pos)!=1):
            #if(current==5 and count==1):
                #sys.exit()

            a = pick_action(pos,epsilon,weights)
            action = (round(math.cos(a)),round(math.sin(a)))
            new_pos = find_new(action,pos)

            #if(current>=1 or find_reward(new_pos)==1):
                #print(pos)

            features = np.empty((DEGREE+1,DEGREE+1))
            next_features = np.empty((DEGREE+1,DEGREE+1))
            reward = find_reward(new_pos)
            for x in range(DEGREE+1):
                for y in range(DEGREE+1):
                    features[x][y] = ((pos[0]/SIZE)**x)*((pos[1]/SIZE)**y)
                    next_features[x][y] = ((new_pos[0]/SIZE)**x)*((new_pos[1]/SIZE)**y)

            old_value = np.sum(np.multiply(weights,features))
            new_value = np.sum(np.multiply(weights,next_features))

            weights = np.add(weights,step_size*(reward+DISCOUNT*new_value-old_value)*features)

            pos = new_pos
            count += 1

            #if(current>=1 and count%1==0):
                #print_weights(10,weights)

        if((current+1)%SHOW_EPISODE==0):
            print(current)
            print_weights(4,weights)
            print(average)
            print(step_size)
            print()
            average=0
        else:
            average = (average*(current%SHOW_EPISODE)+count)/((current+1)%SHOW_EPISODE)
        epsilon = epsilon - epsilon/(EPISODES//2-1)
        step_size = STEP_SIZE/(1+STEP_SIZE_DECAY*current)

    print_episode(weights)


def pick_action(pos,epsilon,weights):
    if(np.random.random()<epsilon):
        return(np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    else:
        return((np.pi/2)*arg_max(pos,weights))

def arg_max(pos,weights):
    values = np.empty(4)
    for i in range(4):
        new_pos = (pos[0]+round(math.cos(np.pi/2*i)),pos[1]+round(math.sin(np.pi/2*i)))
        if((new_pos[0]>=0) and (new_pos[0]<=SIZE-1) and (new_pos[1]>=0) and (new_pos[1]<=SIZE-1)):
            values[i] = find_value(new_pos,weights)
        else:
            values[i] = -math.inf
    return(np.argmax(values))

def find_value(pos,weights):
    temp = 0
    for x in range(DEGREE+1):
        for y in range(DEGREE+1):
            temp += (weights[x][y])*((pos[0]/SIZE)**x)*((pos[1]/SIZE)**y)
    return(temp)

def find_new(action,pos):
    temp1 = action[0]+pos[0]
    if(temp1>=SIZE):
        temp1 = SIZE-1
    elif(temp1<0):
        temp1 = 0
    temp2 = action[1]+pos[1]
    if(temp2>=SIZE):
        temp2 = SIZE-1
    elif(temp2<0):
        temp2 = 0
    return(temp1,temp2)

def find_reward(pos):
    check = False
    for g in GOAL:
        if(g[0]==pos[0] and g[1]==pos[1]):
            check = True
    if(check):
        return 1
    else:
        return -1

def print_weights(d_places,weights):
    print("[", end="")
    for x in range(DEGREE+1):
        for y in range(DEGREE+1):
            if(not((x==DEGREE)and(y==DEGREE))):
                print(round(weights[x][y],d_places), end=", ")
            else:
                print(round(weights[x][y],d_places), end="")
    print("]")

def print_episode(weights):
    for y in range(SIZE):
        line = ""
        for x in range(SIZE):
            a = arg_max((x,y),weights)
            if(a==0):
                line += "> "
            elif(a==1):
                line += "V "
            elif(a==2):
                line += "< "
            else:
                line += "^ "
        print(line+"\n")


main()

99
[-8.2696, -0.1009, -0.6104, -0.4019, 0.2017, 0.1149, -0.5429, 0.2121, 0.1736]
18.707070707070717
0.10101010101010102

199
[-8.386, -0.2016, -0.967, -0.1227, 0.1685, 0.0966, -0.6871, 0.2283, 0.2041]
18.87878787878788
0.06711409395973154

299
[-8.1544, -0.0699, -1.0994, -0.096, 0.1447, 0.0785, -1.001, 0.1915, 0.1937]
17.747474747474744
0.05025125628140704

399
[-8.4089, 0.0206, -1.2147, -0.0105, 0.1311, 0.0648, -1.1837, 0.1603, 0.1877]
14.030303030303031
0.040160642570281124

499
[-8.0946, 0.279, -1.2275, 0.1663, 0.1477, 0.0709, -1.2155, 0.1574, 0.1911]
14.575757575757576
0.033444816053511704

599
[-8.4074, 0.1971, -1.4654, 0.0694, 0.0722, 0.0174, -1.3999, 0.1146, 0.1702]
12.494949494949497
0.028653295128939826

699
[-8.2813, 0.3127, -1.4694, 0.1979, 0.1052, 0.0522, -1.4023, 0.1345, 0.2007]
15.747474747474747
0.02506265664160401

799
[-7.2526, 0.6038, -1.4332, 0.2257, 0.0906, 0.0459, -1.482, 0.1177, 0.1943]
6.686868686868687
0.022271714922048998

899
[-7.638, 0.535, -1.5991, 0.1636, 0